# Atividade 3 — Formação na graduação por sexo e região

Base: **Tabela 3 — Áreas gerais de formação na graduação (IBGE, Censo)**.

A tabela é dividida em 3:
1. homens e mulheres por região com curso superior
2. homens e mulheres por região com curso nas áreas de Ciência, Tecnologia, Engenharias e Matemática (STEM)
3. homens e mulheres por região com curso nas áreas de Educação, Serviços pessoais, Saúde e Bem-estar

O arquivo `Tabela_3_Areas_gerais_de_formacao_na_graduacao.xlsx` precisa estar na mesma pasta.

In [ ]:
# python -m pip install pandas openpyxl
import pandas as pd

In [ ]:
def ler_tabela_ibge(arquivo, nomes_colunas):
    """Lê as tabelas do IBGE pulando o cabeçalho de várias linhas.
    Pega da linha 'Brasil' até a linha 'Distrito Federal'."""
    bruto = pd.read_excel(arquivo, header=None)
    primeira_col = bruto.columns[0]
    inicio = bruto.index[bruto[primeira_col] == "Brasil"][0]
    fim = bruto.index[bruto[primeira_col] == "Distrito Federal"][0]
    df = bruto.loc[inicio:fim, bruto.columns[: len(nomes_colunas)]].copy()
    df.columns = nomes_colunas
    df = df.reset_index(drop=True)
    for c in nomes_colunas[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

REGIOES = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]

In [ ]:
colunas = [
    "uf_regiao",
    "total", "total_homens", "total_mulheres",
    "stem_total", "stem_homens", "stem_mulheres",
    "saude_total", "saude_homens", "saude_mulheres",
]
tabela3 = ler_tabela_ibge("Tabela_3_Areas_gerais_de_formacao_na_graduacao.xlsx", colunas)
tabela3.head()

## Dividindo em 3 tabelas (Brasil + Grandes Regiões)

In [ ]:
por_regiao = tabela3[tabela3["uf_regiao"].isin(["Brasil"] + REGIOES)]

formacao_total = por_regiao[["uf_regiao", "total_homens", "total_mulheres"]].reset_index(drop=True)
formacao_stem = (
    por_regiao[["uf_regiao", "stem_homens", "stem_mulheres"]]
    .rename(columns={"stem_homens": "homens", "stem_mulheres": "mulheres"})
    .reset_index(drop=True)
)
formacao_saude = (
    por_regiao[["uf_regiao", "saude_homens", "saude_mulheres"]]
    .rename(columns={"saude_homens": "homens", "saude_mulheres": "mulheres"})
    .reset_index(drop=True)
)

### 1. Curso superior (total)

In [ ]:
formacao_total

### 2. Ciência, Tecnologia, Engenharias e Matemática

In [ ]:
formacao_stem

### 3. Educação, Serviços pessoais, Saúde e Bem-estar

In [ ]:
formacao_saude

## Estatísticas com `describe()` (somente as 5 Grandes Regiões)

In [ ]:
sem_brasil = lambda df: df[df["uf_regiao"] != "Brasil"]
print("CURSO SUPERIOR (TOTAL)")
sem_brasil(formacao_total).describe()

In [ ]:
print("STEM")
sem_brasil(formacao_stem).describe()

In [ ]:
print("EDUCAÇÃO / SAÚDE")
sem_brasil(formacao_saude).describe()

## Participação das mulheres em cada área (%)

In [ ]:
participacao = pd.DataFrame({
    "uf_regiao": formacao_total["uf_regiao"],
    "%_mulheres_total": (formacao_total["total_mulheres"] / (formacao_total["total_homens"] + formacao_total["total_mulheres"]) * 100).round(1),
    "%_mulheres_stem": (formacao_stem["mulheres"] / (formacao_stem["homens"] + formacao_stem["mulheres"]) * 100).round(1),
    "%_mulheres_saude": (formacao_saude["mulheres"] / (formacao_saude["homens"] + formacao_saude["mulheres"]) * 100).round(1),
})
participacao

## Conclusão

As mulheres são maioria entre as pessoas com curso superior (cerca de 59% no Brasil) e são grande maioria em Educação, Saúde e Bem-estar (perto de 79%). Em Ciência, Tecnologia, Engenharias e Matemática acontece o contrário: elas são só cerca de 34%. Esse padrão se repete em todas as regiões.